# Lesson 9: Checkpointing & Human-in-the-Loop

## WHY

The moderation graph you built in Lesson 8 is **stateless** — every `.invoke()` starts completely
fresh. It remembers nothing between runs, and it acts on every flag without any human oversight.

Both problems matter in production:

**Statelessness breaks real-world use cases.**
A user who says "I think I was wrongly banned" and then "can you explain why?" is having a
*conversation*. Without memory, the bot treats each message as if the previous one never happened.

**Autonomous action is risky.**
Your triage LLM is not perfect. It will have false positives — clean messages that look like
spam. If the bot automatically deletes those, you get angry legitimate users and complaints.
A human-review step catches those errors before any action is taken.

This lesson solves both problems using two LangGraph features:

1. **Checkpointing** — persist graph state to a store keyed by `thread_id`. Every invoke on
   the same thread sees and accumulates the previous state.
2. **Human-in-the-loop (HITL)** — `interrupt()` pauses the graph mid-execution, surfaces a
   value to you, and waits for your decision before continuing.

**By the end of this lesson you will:**
1. Add `MemorySaver` checkpointing to the Lesson 8 graph in one line
2. Use `thread_id` to accumulate state across multiple invocations
3. Inspect paused graph state with `get_state()` and `get_state_history()`
4. Use `interrupt()` to pause before taking a moderation action
5. Resume a paused graph with `Command(resume=...)`
6. Use `interrupt_before=` on `create_agent` as a shortcut for tool-using agents
7. Package everything as an importable factory

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print('API key loaded:', 'OPENAI_API_KEY' in os.environ)

## WHAT — Checkpointing

A **checkpointer** is a storage backend that saves a full snapshot of graph state after every
node execution. LangGraph ships two built-in checkpointers:

| Checkpointer | Import | Use case |
|---|---|---|
| `MemorySaver` | `langgraph.checkpoint.memory` | Development, testing, notebooks |
| `InMemorySaver` | `langgraph.checkpoint.memory` | Same as above (alias) |
| `SqliteSaver` | `langgraph.checkpoint.sqlite` | Production (process-persistent) |
| `AsyncSqliteSaver` | `langgraph.checkpoint.sqlite.aio` | Production async |

You enable checkpointing at **compile time** — one extra argument to `.compile()`:

```python
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)
```

After compiling, you pass a `thread_id` in the `config` on every invoke. LangGraph uses it as
the storage key — all invokes with the same `thread_id` share the same accumulated state.

```python
config = {'configurable': {'thread_id': 'user-12345'}}
graph.invoke(state, config)   # first message for this user
graph.invoke(state, config)   # second message — sees accumulated state
```

`MemorySaver` lives in RAM — data is lost when the process exits. It is ideal for this notebook
and for writing tests. Swap it for `SqliteSaver` in production with no other code changes.

In [ ]:
# Rebuild the Lesson 8 graph (copied here so this notebook is self-contained)

from typing import TypedDict, Annotated, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


# ── State ─────────────────────────────────────────────────────────────────────

class ModerationState(TypedDict):
    messages: Annotated[list, add_messages]  # reducer: appends, never replaces
    verdict: str
    reason: str
    confidence: float
    action_taken: str


# ── Pydantic output models ────────────────────────────────────────────────────

class TriageResult(BaseModel):
    verdict: Literal['clean', 'flagged']
    reason: str = Field(description='Brief explanation of the decision')
    confidence: float = Field(ge=0.0, le=1.0, description='Certainty score, 0 to 1')


class ModerationDecision(BaseModel):
    action: Literal['delete', 'warn', 'timeout', 'none']
    explanation: str = Field(description='Why this action was chosen')


# ── LLMs ─────────────────────────────────────────────────────────────────────

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
triage_llm = llm.with_structured_output(TriageResult)
moderate_llm = llm.with_structured_output(ModerationDecision)

TRIAGE_SYSTEM_PROMPT = (
    'You are a Discord moderation assistant. '
    'Analyze the message for policy violations: spam, hate speech, '
    'harassment, NSFW content, or unsolicited self-promotion. '
    'Respond with a structured verdict.'
)

MODERATION_SYSTEM_PROMPT = (
    'You are a Discord moderator deciding the right action for a flagged message. '
    'Choose the minimum necessary intervention: '
    'delete for clear violations (spam, slurs, NSFW), '
    'warn for borderline content or apparent first offenses, '
    'timeout for repeated violations or serious harassment, '
    'none if on reflection the message does not warrant action.'
)


# ── Nodes ─────────────────────────────────────────────────────────────────────

def triage_node(state: ModerationState) -> dict:
    msg = state['messages'][-1].content if state['messages'] else ''
    result: TriageResult = triage_llm.invoke([
        SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
        HumanMessage(content=f'Message to analyze: {msg!r}'),
    ])
    return {'verdict': result.verdict, 'reason': result.reason, 'confidence': result.confidence}


def moderate_node(state: ModerationState) -> dict:
    msg = state['messages'][-1].content if state['messages'] else ''
    result: ModerationDecision = moderate_llm.invoke([
        SystemMessage(content=MODERATION_SYSTEM_PROMPT),
        HumanMessage(content=f'Flagged: {msg!r} | Reason: {state["reason"]} | Confidence: {state["confidence"]:.0%}'),
    ])
    return {'action_taken': f'{result.action}: {result.explanation}'}


def passthrough_node(state: ModerationState) -> dict:
    return {'action_taken': 'none'}


def route_verdict(state: ModerationState) -> Literal['moderate', 'passthrough']:
    return 'moderate' if state['verdict'] == 'flagged' else 'passthrough'


print('Lesson 8 components loaded.')

## HOW — Adding a Checkpointer

Compare the Lesson 8 compile call against the checkpointed version — the only difference is
one argument:

```python
# Lesson 8 — stateless
graph = builder.compile()

# Lesson 9 — persistent per thread_id
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)
```

Every call to `.invoke()` or `.stream()` now requires `config` with a `thread_id`. LangGraph
will raise an error if you omit it when a checkpointer is attached.

In [ ]:
# Build the graph with a checkpointer
def _build(checkpointer=None):
    b = StateGraph(ModerationState)
    b.add_node('triage', triage_node)
    b.add_node('moderate', moderate_node)
    b.add_node('passthrough', passthrough_node)
    b.add_edge(START, 'triage')
    b.add_conditional_edges('triage', route_verdict)
    b.add_edge('moderate', END)
    b.add_edge('passthrough', END)
    return b.compile(checkpointer=checkpointer)


checkpointer = MemorySaver()
stateful_graph = _build(checkpointer=checkpointer)

print('Stateful graph compiled. Type:', type(stateful_graph).__name__)
print('Checkpointer:', type(checkpointer).__name__)

## HOW — Cross-Invocation Persistence

The key concept: **messages accumulate within the same `thread_id`** because of the
`add_messages` reducer you defined in Lesson 8. Each new `.invoke()` on the same thread
appends to the existing message list rather than replacing it.

Different `thread_id` values are completely independent — they get their own state.

In [ ]:
def make_state(message: str) -> dict:
    return {
        'messages': [HumanMessage(content=message)],
        'verdict': '',
        'reason': '',
        'confidence': 0.0,
        'action_taken': '',
    }


# thread_id 'user-alice' accumulates all three messages
alice_config = {'configurable': {'thread_id': 'user-alice'}}

r1 = stateful_graph.invoke(make_state('Hello, everyone!'), alice_config)
r2 = stateful_graph.invoke(make_state('Can anyone help with Python basics?'), alice_config)

print('After 2 invokes in the same thread:')
print(f'  Total messages in state: {len(r2["messages"])}')
print(f'  Messages: {[m.content for m in r2["messages"]]}')
print()

# A different thread_id starts completely fresh
bob_config = {'configurable': {'thread_id': 'user-bob'}}
r_bob = stateful_graph.invoke(make_state('Hello from Bob!'), bob_config)

print('Bob is on a separate thread:')
print(f'  Messages in Bob thread: {len(r_bob["messages"])}')

## HOW — Inspecting State: `get_state()` and `get_state_history()`

Once a checkpointer is attached, two new graph methods become useful:

| Method | What it returns |
|--------|-----------------|
| `graph.get_state(config)` | The **current** snapshot for this thread |
| `graph.get_state_history(config)` | **All** snapshots (most recent first) |

A snapshot has:
- `.values` — the full state dict at that point
- `.next` — which nodes will execute next (empty tuple = graph finished)
- `.tasks` — pending tasks, including any interrupt payloads
- `.config` — the full config with checkpoint ID embedded

In [ ]:
# Inspect Alice's current state
alice_state = stateful_graph.get_state(alice_config)

print('Current state for thread "user-alice":')
print(f'  next nodes:    {alice_state.next}')   # () = finished
print(f'  message count: {len(alice_state.values["messages"])}')
print(f'  last verdict:  {alice_state.values["verdict"]}')
print(f'  action taken:  {alice_state.values["action_taken"]}')
print()

print('State history (most recent first):')
for i, snap in enumerate(stateful_graph.get_state_history(alice_config)):
    n_msgs = len(snap.values.get('messages', []))
    verdict = snap.values.get('verdict', '-')
    print(f'  [{i}] msgs={n_msgs}, verdict={verdict!r:10}, next={snap.next}')

## WHAT — Human-in-the-Loop with `interrupt()`

`interrupt()` is a function you call **inside a node** to pause the graph and hand control
back to the caller. The caller can inspect the situation, make a decision, then resume.

```python
from langgraph.types import interrupt, Command

def human_review_node(state: ModerationState) -> dict:
    # This pauses the graph and sends the dict to the caller.
    # The return value is whatever the caller passes in Command(resume=...).
    decision = interrupt({
        'verdict': state['verdict'],
        'reason': state['reason'],
        'message': state['messages'][-1].content,
        'question': 'Approve this action?',
    })
    # Execution continues here only after the graph is resumed.
    return {'moderator_override': decision}
```

**The re-execution rule.** When the graph is resumed, LangGraph re-runs the entire node from
the top. On the re-run, `interrupt()` does *not* pause again — it immediately returns the
resume value. This means any logic before the `interrupt()` call runs twice. Keep nodes
side-effect-free before the interrupt point.

**Requirements:**
- A **checkpointer must be attached** — `interrupt()` relies on saved state to pause and resume
- Resume with `graph.invoke(Command(resume=your_value), same_config)`

**What the caller sees:**
- `.invoke()` — returns state dict with an extra `__interrupt__` key containing the payload
- `.stream()` — yields a `{'__interrupt__': (...)}` event before stopping

In [ ]:
# Minimal interrupt() demo — no LLM, just shows the pause/resume pattern clearly

from langgraph.types import interrupt, Command


class SimpleState(TypedDict):
    value: str
    approved: bool


def work_node(state: SimpleState) -> dict:
    print('  [work] doing some work...')
    return {'value': 'computed result'}


def review_node(state: SimpleState) -> dict:
    print('  [review] pausing for human input...')
    # Pause here. The dict passed to interrupt() is the payload surfaced to the caller.
    human_answer = interrupt({'value': state['value'], 'question': 'Approve this?'})
    print(f'  [review] resumed! human said: {human_answer!r}')
    return {'approved': human_answer == 'yes'}


demo_builder = StateGraph(SimpleState)
demo_builder.add_node('work', work_node)
demo_builder.add_node('review', review_node)
demo_builder.add_edge(START, 'work')
demo_builder.add_edge('work', 'review')
demo_builder.add_edge('review', END)

demo_checkpointer = MemorySaver()
demo_graph = demo_builder.compile(checkpointer=demo_checkpointer)
demo_config = {'configurable': {'thread_id': 'demo-1'}}

print('=== First invoke: runs until interrupt() ===\n')
paused = demo_graph.invoke({'value': '', 'approved': False}, demo_config)
print()
print('Interrupt payload:', paused['__interrupt__'][0].value)

In [ ]:
print('=== Resume with Command(resume=...) ===\n')
# Pass Command(resume=your_answer) as the input — graph re-runs review_node,
# interrupt() now returns 'yes' immediately instead of pausing.
final = demo_graph.invoke(Command(resume='yes'), demo_config)
print()
print('Final state:')
print(f'  value:    {final["value"]}')
print(f'  approved: {final["approved"]}')

## HOW — Add Human Review to the Moderation Graph

Now apply HITL to the real moderation pipeline. Add a `human_review` node between `triage`
and `moderate`. The graph pauses there, a human decides whether to proceed, and execution
continues only when the decision arrives.

**New graph shape:**

```
START
  → triage
     --[flagged]--> human_review   (PAUSES HERE)
                       --[approved]--> moderate  → END
                       --[rejected]--> passthrough → END
     --[clean]---> passthrough → END
```

Notice: `human_review` sits **only** on the `flagged` path. Clean messages skip it entirely.

In [ ]:
class HitlModerationState(TypedDict):
    messages: Annotated[list, add_messages]
    verdict: str
    reason: str
    confidence: float
    action_taken: str
    moderator_decision: Optional[str]   # set by the human reviewer


def human_review_node(state: HitlModerationState) -> dict:
    msg = state['messages'][-1].content if state['messages'] else ''
    # Pause and surface context to the moderator.
    # The return value of interrupt() is whatever the moderator sends in Command(resume=...).
    decision = interrupt({
        'message':    msg,
        'verdict':    state['verdict'],
        'reason':     state['reason'],
        'confidence': f"{state['confidence']:.0%}",
        'question':   "Approve action? Reply 'approve' or 'reject'.",
    })
    return {'moderator_decision': decision}


def route_after_review(
    state: HitlModerationState,
) -> Literal['moderate', 'passthrough']:
    # If the moderator approved, take action; otherwise let it pass.
    return 'moderate' if state.get('moderator_decision') == 'approve' else 'passthrough'


print('human_review_node and route_after_review defined.')

In [ ]:
# Nodes for HitlModerationState (same logic, different TypedDict type annotation)

def triage_node_hitl(state: HitlModerationState) -> dict:
    msg = state['messages'][-1].content if state['messages'] else ''
    result: TriageResult = triage_llm.invoke([
        SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
        HumanMessage(content=f'Message to analyze: {msg!r}'),
    ])
    return {'verdict': result.verdict, 'reason': result.reason, 'confidence': result.confidence}


def moderate_node_hitl(state: HitlModerationState) -> dict:
    msg = state['messages'][-1].content if state['messages'] else ''
    result: ModerationDecision = moderate_llm.invoke([
        SystemMessage(content=MODERATION_SYSTEM_PROMPT),
        HumanMessage(content=f'Flagged: {msg!r} | Reason: {state["reason"]} | Confidence: {state["confidence"]:.0%}'),
    ])
    return {'action_taken': f'{result.action}: {result.explanation}'}


def passthrough_node_hitl(state: HitlModerationState) -> dict:
    return {'action_taken': 'none'}


def route_verdict_hitl(
    state: HitlModerationState,
) -> Literal['human_review', 'passthrough']:
    return 'human_review' if state['verdict'] == 'flagged' else 'passthrough'


# Assemble the HITL graph
hitl_checkpointer = MemorySaver()

hitl_builder = StateGraph(HitlModerationState)
hitl_builder.add_node('triage',       triage_node_hitl)
hitl_builder.add_node('human_review', human_review_node)
hitl_builder.add_node('moderate',     moderate_node_hitl)
hitl_builder.add_node('passthrough',  passthrough_node_hitl)

hitl_builder.add_edge(START, 'triage')
hitl_builder.add_conditional_edges('triage',       route_verdict_hitl)
hitl_builder.add_conditional_edges('human_review', route_after_review)
hitl_builder.add_edge('moderate',    END)
hitl_builder.add_edge('passthrough', END)

hitl_graph = hitl_builder.compile(checkpointer=hitl_checkpointer)

print('HITL moderation graph compiled!')
print()
print(hitl_graph.get_graph().draw_mermaid())

## HOW — Streaming Through an Interrupt

Using `.stream()` makes the interrupt visible as a distinct event in the stream, which is
cleaner than `.invoke()` for interactive moderation workflows. The pattern:

1. Stream the initial invocation — iterate until you see `__interrupt__`
2. Read the interrupt payload and make a decision
3. Stream again with `Command(resume=decision)` and the same `config`

In [ ]:
spam_message = 'Check out my crypto trading course — 99% win rate! DM me now!!!'
hitl_config = {'configurable': {'thread_id': 'hitl-demo-1'}}

initial_state = {
    'messages':           [HumanMessage(content=spam_message)],
    'verdict':            '',
    'reason':             '',
    'confidence':         0.0,
    'action_taken':       '',
    'moderator_decision': None,
}

print('=== Phase 1: streaming until interrupt ===\n')
interrupt_payload = None

for event in hitl_graph.stream(initial_state, hitl_config):
    for node_name, output in event.items():
        if node_name == '__interrupt__':
            # output is a tuple of Interrupt objects
            interrupt_payload = output[0].value
            print('[PAUSED — awaiting human decision]')
            print('Interrupt payload:')
            for k, v in interrupt_payload.items():
                print(f'  {k}: {v}')
        else:
            verdict = output.get('verdict', '')
            label = f'verdict={verdict!r}' if verdict else str(output)
            print(f'[{node_name}] {label}')

In [ ]:
# Inspect the paused state before deciding
paused_state = hitl_graph.get_state(hitl_config)
print('Graph is paused.')
print('next nodes:', paused_state.next)     # ('human_review',)
print('verdict:   ', paused_state.values['verdict'])
print('reason:    ', paused_state.values['reason'])

In [ ]:
print('=== Phase 2: moderator approves, resume graph ===\n')

# Simulate moderator deciding to approve the action.
# Same config — LangGraph looks up the saved checkpoint for this thread_id.
for event in hitl_graph.stream(Command(resume='approve'), hitl_config):
    for node_name, output in event.items():
        if output:
            print(f'[{node_name}]', output)

print()
final_state = hitl_graph.get_state(hitl_config)
print('Final action taken:', final_state.values['action_taken'])

In [ ]:
# Show what happens when the moderator REJECTS the AI recommendation
print('=== Test: moderator rejects on a new thread ===\n')

reject_config = {'configurable': {'thread_id': 'hitl-demo-reject'}}

# Phase 1: run until interrupt
hitl_graph.invoke(initial_state, reject_config)

# Phase 2: moderator says 'reject' — message should pass through
result = hitl_graph.invoke(Command(resume='reject'), reject_config)

print('action_taken:', result['action_taken'])   # 'none' — passthrough
print('moderator_decision:', result['moderator_decision'])   # 'reject'

## WHAT — `interrupt_before=` on `create_agent`

If you are using `create_agent()` (the ReAct loop) instead of a manual `StateGraph`,
you get two shortcut parameters that add HITL without any graph wiring:

| Parameter | Pauses | Use case |
|-----------|--------|----------|
| `interrupt_before=['tools']` | Before tool execution | Review what tool the agent wants to call |
| `interrupt_after=['tools']` | After tool execution | Review what the tool returned |

The node names `'model'` and `'tools'` come from the ReAct graph's internal structure,
which you can inspect with `.get_graph(xray=True).draw_mermaid()`.

**This approach requires a `checkpointer`** — either passed to `create_agent()` directly or
to `.compile()` (they are the same call under the hood).

```python
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

agent = create_agent(
    'openai:gpt-4o-mini',
    tools=[my_tool],
    system_prompt='...',
    checkpointer=MemorySaver(),
    interrupt_before=['tools'],   # pause before every tool call
)
```

Resume the same way: `agent.invoke(Command(resume=None), config)`. Passing `None` tells the
agent to proceed — since the interrupt happens *before* the node, no return value is needed.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def ban_user(user_id: str, reason: str) -> str:
    '''Ban a user from the server.

    Parameters
    ----------
    user_id : str
        Discord user ID to ban.
    reason : str
        Reason for the ban, logged for audit purposes.
    '''
    # Stub — in production this would call a Discord API.
    return f'User {user_id} banned. Reason: {reason}'


tool_review_checkpointer = MemorySaver()

tool_review_agent = create_agent(
    llm,
    tools=[ban_user],
    system_prompt='You are a Discord moderation assistant. Use ban_user when a message warrants a ban.',
    checkpointer=tool_review_checkpointer,
    interrupt_before=['tools'],   # pause before executing any tool
)

print('Agent with interrupt_before=["tools"] compiled.')
print()
print('Internal graph (notice the interrupt annotation on tools node):')
print(tool_review_agent.get_graph(xray=True).draw_mermaid())

In [ ]:
from langchain_core.messages import HumanMessage

agent_config = {'configurable': {'thread_id': 'tool-review-1'}}

print('=== Phase 1: agent decides to ban, pauses before tool call ===\n')
paused = tool_review_agent.invoke(
    {'messages': [HumanMessage(content='User 98765 is spamming crypto links — please ban them.')]},
    agent_config,
)

# The graph paused before the tool executed — inspect what the agent wanted to do.
pending_state = tool_review_agent.get_state(agent_config)
last_ai_msg = pending_state.values['messages'][-1]

print('Agent wants to call:')
if hasattr(last_ai_msg, 'tool_calls') and last_ai_msg.tool_calls:
    for tc in last_ai_msg.tool_calls:
        print(f'  tool: {tc["name"]}')
        print(f'  args: {tc["args"]}')
print()
print('Next node:', pending_state.next)

In [ ]:
print('=== Phase 2: moderator approves, tool executes ===\n')
# Pass None as resume value — interrupt_before pauses don't need a return value.
final = tool_review_agent.invoke(Command(resume=None), agent_config)

# Find the tool result message
from langchain_core.messages import ToolMessage
tool_results = [m for m in final['messages'] if isinstance(m, ToolMessage)]
if tool_results:
    print('Tool result:', tool_results[-1].content)
else:
    print('Final messages:', [type(m).__name__ for m in final['messages']])

## WHAT — Choosing Between Approaches

| Scenario | Best approach |
|----------|---------------|
| ReAct agent (create_agent), pause before tools | `interrupt_before=['tools']` on `create_agent` |
| Custom StateGraph, pause at a specific node | `interrupt()` inside that node |
| Need to pass a value back from the human | `interrupt()` + `Command(resume=value)` |
| Just approve/deny before a node | `interrupt_before=` + `Command(resume=None)` |
| Intercept/modify agent messages without pausing | `middleware=` on `create_agent` |

The `middleware=` parameter accepts `AgentMiddleware` instances — a more advanced API for
intercepting, logging, or modifying messages at every step without needing an explicit
node or interrupt. It is covered in Module 8 (Production Patterns).

## HOW — Importable Factory

Package the full HITL moderation graph into a factory that mirrors the Lesson 8 pattern.

In [ ]:
def build_hitl_moderation_graph(
    model: str = 'gpt-4o-mini',
    temperature: float = 0,
    checkpointer=None,
):
    '''Build and compile the 4-node HITL moderation StateGraph.

    Graph shape:
        START → triage
            --[flagged]--> human_review  (interrupts for moderator input)
                            --[approve]--> moderate    → END
                            --[reject]---> passthrough → END
            --[clean]----> passthrough → END

    Parameters
    ----------
    model : str
        OpenAI model name for triage and moderation nodes.
    temperature : float
        Sampling temperature. Use 0 for deterministic classification.
    checkpointer : langgraph.checkpoint.base.BaseCheckpointSaver or None
        Checkpointer for persistence. Required for interrupt() to work.
        Defaults to a fresh MemorySaver if None.

    Returns
    -------
    langgraph.graph.state.CompiledStateGraph
        Compiled graph. Call .invoke(state, config) with a thread_id.
        Resume after interrupt with .invoke(Command(resume=value), config).
    '''
    if checkpointer is None:
        checkpointer = MemorySaver()

    _llm = ChatOpenAI(model=model, temperature=temperature)
    _triage_llm = _llm.with_structured_output(TriageResult)
    _moderate_llm = _llm.with_structured_output(ModerationDecision)

    def _triage(state: HitlModerationState) -> dict:
        msg = state['messages'][-1].content if state['messages'] else ''
        res: TriageResult = _triage_llm.invoke([
            SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
            HumanMessage(content=f'Message to analyze: {msg!r}'),
        ])
        return {'verdict': res.verdict, 'reason': res.reason, 'confidence': res.confidence}

    def _human_review(state: HitlModerationState) -> dict:
        msg = state['messages'][-1].content if state['messages'] else ''
        decision = interrupt({
            'message':    msg,
            'verdict':    state['verdict'],
            'reason':     state['reason'],
            'confidence': f"{state['confidence']:.0%}",
            'question':   "Reply 'approve' or 'reject'.",
        })
        return {'moderator_decision': decision}

    def _moderate(state: HitlModerationState) -> dict:
        msg = state['messages'][-1].content if state['messages'] else ''
        res: ModerationDecision = _moderate_llm.invoke([
            SystemMessage(content=MODERATION_SYSTEM_PROMPT),
            HumanMessage(content=f'Flagged: {msg!r} | Reason: {state["reason"]}'),
        ])
        return {'action_taken': f'{res.action}: {res.explanation}'}

    def _passthrough(state: HitlModerationState) -> dict:
        return {'action_taken': 'none'}

    def _route_triage(s: HitlModerationState) -> Literal['human_review', 'passthrough']:
        return 'human_review' if s['verdict'] == 'flagged' else 'passthrough'

    def _route_review(s: HitlModerationState) -> Literal['moderate', 'passthrough']:
        return 'moderate' if s.get('moderator_decision') == 'approve' else 'passthrough'

    b = StateGraph(HitlModerationState)
    b.add_node('triage',       _triage)
    b.add_node('human_review', _human_review)
    b.add_node('moderate',     _moderate)
    b.add_node('passthrough',  _passthrough)
    b.add_edge(START, 'triage')
    b.add_conditional_edges('triage',       _route_triage)
    b.add_conditional_edges('human_review', _route_review)
    b.add_edge('moderate',    END)
    b.add_edge('passthrough', END)
    return b.compile(checkpointer=checkpointer)


# Smoke-test — compile only, no API calls
g = build_hitl_moderation_graph()
print('build_hitl_moderation_graph() OK. Type:', type(g).__name__)

In [ ]:
# End-to-end test: full approve flow using the factory
graph = build_hitl_moderation_graph()
cfg   = {'configurable': {'thread_id': 'factory-e2e-1'}}
msg   = 'Selling premium Discord nitro at 90% off — limited slots!'

state = {
    'messages': [HumanMessage(content=msg)], 'verdict': '', 'reason': '',
    'confidence': 0.0, 'action_taken': '', 'moderator_decision': None,
}

# Phase 1 — triage runs, human_review pauses
p = graph.invoke(state, cfg)
payload = p.get('__interrupt__', [None])[0]

if payload:
    print('PAUSED for review:')
    for k, v in payload.value.items():
        print(f'  {k}: {v}')
    print()

    # Phase 2 — approve and complete
    final = graph.invoke(Command(resume='approve'), cfg)
    print('Action taken:', final['action_taken'])
else:
    # Message was clean, no interrupt
    print('Message was clean. Action:', p['action_taken'])

## Summary

| Concept | Key API |
|---------|--------|
| In-memory persistence | `MemorySaver()` → `builder.compile(checkpointer=...)` |
| Thread isolation | `config = {'configurable': {'thread_id': 'some-unique-id'}}` |
| Cross-invocation accumulation | `add_messages` reducer on `messages` field |
| Inspect current state | `graph.get_state(config)` |
| Inspect full history | `graph.get_state_history(config)` |
| Pause a node | `interrupt(payload_value)` inside the node function |
| Resume after pause | `graph.invoke(Command(resume=answer), same_config)` |
| HITL on create_agent | `interrupt_before=['tools']` / `interrupt_after=['tools']` |
| Approve without return value | `Command(resume=None)` |

## What's Next — Lesson 10: RAG Agent

Your moderation bot can now flag, pause, and wait for a human decision. But your users also
ask questions — about server rules, FAQs, game guides. That is where **Retrieval-Augmented
Generation (RAG)** comes in.

In Lesson 10 you will:
- Embed documents with `OpenAIEmbeddings` and store them in a FAISS vector store
- Wrap the retriever as a LangChain tool with `create_retriever_tool`
- Pass it to `create_agent` to build a Q&A bot that answers from your own knowledge base
- Test retrieval quality with cosine similarity scores